In [5]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. ROBUST MAPPING (Handles Case Sensitivity)
# ==========================================
VAR_MAP = {
    'State_FIPS': ['_STATE', 'STATE', 'FIPS'],
    'Age': ['_AGE80', '_AGE_G', 'AGE'],
    'BMI': ['_BMI5', 'BMI'],
    'Smoker': ['_RFSMOK3', 'SMOKER3', 'SMOKE100'],
    'PhysicalHealthDays': ['PHYSHLTH', 'PHYSHLTH'],
    'MentalHealthDays': ['MENTHLTH', 'MENTHLTH'],
    'Income': ['INCOME2', 'INCOME3', 'INCOME'], 
    'Education': ['EDUCA', 'EDUCA'],
    'HeartAttack': ['CVDINFR4', 'CVDINFR3', 'CVDINFR'],
    'Angina': ['CVDCRHD4', 'CVDCRHD3', 'CVDCRHD'],
    'Stroke': ['CVDSTRK3', 'CVDSTRK'],
    'Asthma': ['ASTHMA3', 'ASTHMA'],
    'SkinCancer': ['CHCSCNCR', 'CHCSCNCR'],
    'OtherCancer': ['CHCOCNCR', 'CHCOCNCR'],
    'COPD': ['CHCCOPD3', 'CHCCOPD2', 'CHCCOPD'],
    'Depression': ['ADDEPEV3', 'ADDEPEV2'],
    'KidneyDisease': ['CHCKDNY2', 'CHCKDNY1'],
    'Diabetes': ['DIABETE4', 'DIABETE3', 'DIABETE2'],
    'Arthritis': ['HAVARTH5', 'HAVARTH4', 'HAVARTH3'],
    'DifficultyWalking': ['DIFFWALK', 'DIFFWALK']
}

FIPS_TO_STATE = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California',
    8: 'Colorado', 9: 'Connecticut', 10: 'Delaware', 11: 'District of Columbia',
    12: 'Florida', 13: 'Georgia', 15: 'Hawaii', 16: 'Idaho', 17: 'Illinois',
    18: 'Indiana', 19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 22: 'Louisiana',
    23: 'Maine', 24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan',
    27: 'Minnesota', 28: 'Mississippi', 29: 'Missouri', 30: 'Montana',
    31: 'Nebraska', 32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey',
    35: 'New Mexico', 36: 'New York', 37: 'North Carolina', 38: 'North Dakota',
    39: 'Ohio', 40: 'Oklahoma', 41: 'Oregon', 42: 'Pennsylvania',
    44: 'Rhode Island', 45: 'South Carolina', 46: 'South Dakota', 47: 'Tennessee',
    48: 'Texas', 49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington',
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def get_col_name(df, possible_names):
    """Case-insensitive search for columns."""
    df_cols_upper = [c.upper() for c in df.columns]
    for name in possible_names:
        if name.upper() in df_cols_upper:
            # Return the actual column name from the dataframe
            return df.columns[df_cols_upper.index(name.upper())]
    return None

def clean_brfss_year(file_path, year):
    print(f"   -> Loading BRFSS {year}...")
    try:
        df = pd.read_csv(file_path, low_memory=False, encoding='cp1252') 
    except FileNotFoundError:
        print(f"      ERROR: File not found: {file_path}")
        return pd.DataFrame()

    clean_data = pd.DataFrame()
    
    # Extract columns dynamically
    found_cols = 0
    for target_name, possible_cols in VAR_MAP.items():
        raw_col = get_col_name(df, possible_cols)
        if raw_col:
            clean_data[target_name] = df[raw_col]
            found_cols += 1
        else:
            # Only print warning if critical targets are missing
            if target_name in ['HeartAttack', 'Diabetes']:
                print(f"      Warning: Missing critical col {target_name} in {year}")

    if found_cols < 5:
        print(f"      CRITICAL: Only found {found_cols} columns. Check CSV headers!")
        print(f"      Available Headers: {list(df.columns)[:10]}...")
        return pd.DataFrame()

    # Standardize Values (1=Yes, 2=No -> 1/0)
    binary_cols = ['HeartAttack', 'Angina', 'Stroke', 'Asthma', 'SkinCancer', 
                   'OtherCancer', 'COPD', 'Depression', 'KidneyDisease', 
                   'Diabetes', 'Arthritis', 'DifficultyWalking']
    
    for col in binary_cols:
        if col in clean_data.columns:
            clean_data = clean_data[clean_data[col].isin([1, 2, 3, 4])] 
            clean_data[col] = clean_data[col].apply(lambda x: 0 if x == 2 else 1)

    if 'BMI' in clean_data.columns:
        clean_data['BMI'] = clean_data['BMI'] / 100
        clean_data = clean_data[clean_data['BMI'] < 100]

    clean_data['Year'] = year
    if 'State_FIPS' in clean_data.columns:
        clean_data['State'] = clean_data['State_FIPS'].map(FIPS_TO_STATE)
        clean_data.drop(columns=['State_FIPS'], inplace=True)

    return clean_data.dropna()

def clean_epa_year(file_path, year):
    """Aggregates EPA Air Quality Data."""
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"      Warning: EPA file not found: {file_path}")
        return pd.DataFrame()

    df.columns = [c.lower() for c in df.columns]
    
    # Look for correct columns based on common EPA formats
    state_col = next((c for c in df.columns if 'state' in c and 'code' not in c), None)
    aqi_col = next((c for c in df.columns if 'aqi' in c or 'mean' in c), None)

    if state_col and aqi_col:
        # Check if column is numeric before grouping
        if df[aqi_col].dtype == object:
             df[aqi_col] = pd.to_numeric(df[aqi_col], errors='coerce')
             
        epa_agg = df.groupby(state_col)[aqi_col].mean().reset_index()
        epa_agg.columns = ['State', 'Avg_AQI']
        return epa_agg
    else:
        return pd.DataFrame()

# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================
years_to_process = [2020, 2022, 2023]
merged_dfs = []

print("--- Starting Multi-Year Data Fusion ---")

for year in years_to_process:
    print(f"\nProcessing Year: {year}")
    
    # FIX 1: Updated filename patterns to match your screenshot
    if year == 2023:
        brfss_path = f'data/raw/BRFSS{year}.csv' # Uppercase for 2023 based on screenshot? 
        # Or try lowercase if that fails: 'data/raw/brfss2023.csv'
    else:
        brfss_path = f'data/raw/brfss{year}.csv'
        
    epa_path = f'data/raw/annual_aqi_by_county_{year}.csv' # MATCHES SCREENSHOT
    
    df_health = clean_brfss_year(brfss_path, year)
    df_epa = clean_epa_year(epa_path, year)
    
    if not df_health.empty and not df_epa.empty:
        df_merged = pd.merge(df_health, df_epa, on='State', how='left')
        df_merged['Avg_AQI'] = df_merged['Avg_AQI'].fillna(df_merged['Avg_AQI'].mean())
        merged_dfs.append(df_merged)
        print(f"   -> Success: {len(df_merged)} rows processed.")
    elif not df_health.empty:
        print("   -> EPA missing, using Health only.")
        merged_dfs.append(df_health)

if merged_dfs:
    final_brfss = pd.concat(merged_dfs, ignore_index=True)
    # Create the folder if it doesn't exist
    os.makedirs('data/processed', exist_ok=True)
    final_brfss.to_csv('data/processed/multi_year_health_data.csv', index=False)
    print(f"\nSUCCESS: Combined Data Saved ({len(final_brfss)} rows).")

# ==========================================
# 4. FIX AUTISM DATA (The Token Error)
# ==========================================
print("\n--- Processing Specialized Datasets ---")

# Autism Fix
try:
    aut_path = 'data/raw/Autism-Adult-Data.arff'
    if os.path.exists(aut_path):
        # FIX 2: Try reading with standard CSV, if fails, manual parse
        try:
            # Often ARFF has 20 lines of comments. We skip them? 
            # Better approach: Read as text, find @data, read rest.
            with open(aut_path, 'r') as f:
                lines = f.readlines()
            
            data_start = 0
            for i, line in enumerate(lines):
                if '@data' in line.lower():
                    data_start = i + 1
                    break
            
            if data_start > 0:
                from io import StringIO
                data_str = "".join(lines[data_start:])
                df_aut = pd.read_csv(StringIO(data_str), header=None)
                print("   -> Autism ARFF loaded successfully.")
                df_aut.to_csv('data/processed/autism_clean.csv', index=False)
            else:
                # Try simple read if @data not found
                df_aut = pd.read_csv(aut_path, on_bad_lines='skip')
                print("   -> Autism loaded (basic mode).")

        except Exception as e:
            print(f"   -> Autism Read Error: {e}")
            
except Exception as e:
    print(f"Global Error: {e}")

--- Starting Multi-Year Data Fusion ---

Processing Year: 2020
   -> Loading BRFSS 2020...
   -> Success: 337842 rows processed.

Processing Year: 2022
   -> Loading BRFSS 2022...
   -> Success: 369050 rows processed.

Processing Year: 2023
   -> Loading BRFSS 2023...
   -> Success: 367297 rows processed.

SUCCESS: Combined Data Saved (1074189 rows).

--- Processing Specialized Datasets ---
   -> Autism ARFF loaded successfully.
